# Гамма-нож -- работа над ошибками, исследование
Честная валидация, PR-AUC, сравнение с исходной версией.

## Данные

In [ ]:
!kaggle competitions download -c gamma-knife-3
import zipfile, os
os.makedirs("gamma-knife-3", exist_ok=True)
with zipfile.ZipFile("gamma-knife-3.zip") as z:
    z.extractall("gamma-knife-3")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, balanced_accuracy_score
from catboost import CatBoostClassifier
import lightgbm as lgb
import xgboost as xgb

SEED = 42
N_JOBS = 16
CAT_COLS = ["Онкологический диагноз", "Лекарственное лечение"]
DATES = ["Дата рождения",
         "Дата постановки онкологического диагноза / начала первичного лечения",
         "Дата удаления первичного очага", "Дата развития МГМ",
         "Дата проведения ОВГМ", "Дата операции на ГМ", "Дата 1-ой РХ"]

In [ ]:
df_train = pd.read_csv("gamma-knife-3/train.csv")
df_test = pd.read_csv("gamma-knife-3/test.csv")
test_ids = df_test["ID"].copy()

## Подготовка
Одна функция на train и test - расхождения между ними невозможны по построению

In [ ]:
def to_float(s):
    return s.astype(str).str.replace(",", ".", regex=False).astype(float)

def flag(s):
    v = s.astype(str).str.strip().str.lower()
    return (~v.isin(["nan", "none", ""]) & ~v.eq("нет")).astype(int)

def prepare(df):
    df = df.copy()
    for d in DATES:
        df[d] = pd.to_datetime(df[d], format="%d.%m.%Y", errors="coerce")

    df["Мужчина"] = df["Пол"].str.upper().eq("М").astype(int)
    df["Возраст"] = (df["Дата 1-ой РХ"] - df["Дата рождения"]).dt.days / 365.25
    df["Время_метастазирования"] = (df["Дата развития МГМ"] - df["Дата постановки онкологического диагноза / начала первичного лечения"]).dt.days
    df["Время_реагирования"] = (df["Дата 1-ой РХ"] - df["Дата развития МГМ"]).dt.days
    df["ОВГМ"] = df["Дата проведения ОВГМ"].notna().astype(int)
    df["Операция"] = df["Дата операции на ГМ"].notna().astype(int)
    df["Экстракраниальные метастазы"] = flag(df["Экстракраниальные метастазы"])

    df["Суммарный объем очагов"] = to_float(df["Суммарный объем очагов"])
    df["Объем на очаг"] = df["Суммарный объем очагов"] / df["Число очагов в ГМ"]
    df["Скорость_метастазирования"] = df["Суммарный объем очагов"] / df["Время_метастазирования"]
    df["Скорость_реагирования"] = df["Суммарный объем очагов"] / (df["Время_реагирования"] + 1)
    df["Индекс_прогрессирования"] = df["Время_метастазирования"] - df["Время_реагирования"]
    df["Возраст * число очагов"] = df["Возраст"] * df["Число очагов в ГМ"]

    for c in CAT_COLS:
        df[c] = df[c].fillna("Unknown").astype(str)

    drop = DATES + ["Пол", "Объем максимального очага", "ID"]
    df = df.drop(columns=[c for c in drop if c in df.columns])
    return df.replace([np.inf, -np.inf], np.nan)

In [ ]:
y_raw = df_train["Интракраниальная прогрессия"]
mask = y_raw.notna()
y = y_raw[mask].isin(["ДМ", "ЛР+ДМ", "ЛР"]).astype(int)

X = prepare(df_train[mask].drop(columns=["Интракраниальная прогрессия", "Активирующие мутации",
                                         "Дистантные метастазы", "Локальный рецидив"], errors="ignore"))
T = prepare(df_test)

keep = X["Время_метастазирования"].notna() & X["Время_реагирования"].notna() & (X["Время_реагирования"] >= 0)
X, y = X[keep].reset_index(drop=True), y[keep].reset_index(drop=True)
T = T.reindex(columns=X.columns)

print(X.shape, T.shape, round(y.mean(), 4))

## Винзоризация внутри фолда
В первой версии квантили считались по всему train до разбиения. Здесь порог берется только из обучающей части.

In [ ]:
WINSOR = ["Время_метастазирования", "Время_реагирования", "Суммарный объем очагов", "Объем на очаг"]

def fit_caps(df):
    return {c: df[c].quantile(0.99) for c in WINSOR if c in df}

def apply_caps(df, caps):
    df = df.copy()
    for c, cap in caps.items():
        df[c] = df[c].clip(lower=0, upper=cap)
    for c in WINSOR:
        if c in df:
            df[c] = np.log1p(df[c])
    return df

## Обучение
Порог подбираем на внутренних фолдах обучающей части и применяем к отложенной (в прошлой версии он не обращался к отложенной части)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression


def make_lr():
    """Базовая модель для сравнения. Бустингу не нужен препроцессинг,
    линейной модели нужен: пропуски, масштаб, категории в числа."""
    num = [c for c in X.columns if c not in CAT_COLS]
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                              ("sc", StandardScaler())]), num),
            ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=10), CAT_COLS),
        ])),
        ("lr", LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0, random_state=SEED)),
    ])


def fit_predict(kind, Xtr, ytr, Xva, params=None):
    if kind == "cb":
        m = CatBoostClassifier(**(params or {}), random_seed=SEED, verbose=False,
                               task_type="CPU", thread_count=N_JOBS)
        m.fit(Xtr, ytr, cat_features=CAT_COLS)
    elif kind == "lr":
        m = make_lr()
        m.fit(Xtr, ytr)
    elif kind == "lgb":
        Xtr, Xva = Xtr.copy(), Xva.copy()
        for c in CAT_COLS:
            cats = Xtr[c].astype("category").cat.categories
            Xtr[c] = pd.Categorical(Xtr[c], categories=cats)
            Xva[c] = pd.Categorical(Xva[c], categories=cats)
        m = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.02, num_leaves=63,
                               min_child_samples=20, subsample=0.9, colsample_bytree=0.9,
                               random_state=SEED, verbose=-1, n_jobs=N_JOBS)
        m.fit(Xtr, ytr)
    else:
        Xtr, Xva = Xtr.copy(), Xva.copy()
        for c in CAT_COLS:
            cats = Xtr[c].astype("category").cat.categories
            Xtr[c] = pd.Categorical(Xtr[c], categories=cats)
            Xva[c] = pd.Categorical(Xva[c], categories=cats)
        m = xgb.XGBClassifier(n_estimators=800, learning_rate=0.05, max_depth=6,
                              subsample=0.8, colsample_bytree=0.8, enable_categorical=True,
                              tree_method="hist", random_state=SEED, eval_metric="logloss",
                              n_jobs=N_JOBS)
        m.fit(Xtr, ytr)
    return m, m.predict_proba(Xva)[:, 1]


def pick_threshold(p, t):
    grid = np.linspace(0.05, 0.95, 181)
    return float(grid[np.argmax([balanced_accuracy_score(t, (p >= g).astype(int)) for g in grid])])

In [ ]:
from sklearn.isotonic import IsotonicRegression

CB_PARAMS = dict(iterations=1000, learning_rate=0.025, depth=7, l2_leaf_reg=11,
                 min_data_in_leaf=51, random_strength=0.907, subsample=0.849,
                 bootstrap_type="Bernoulli", loss_function="Logloss")


def evaluate(kind, params=None, n_splits=5, seed=SEED, calibrate=False):
    outer = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    rows, oof = [], np.zeros(len(X))

    for fold, (tr, va) in enumerate(outer.split(X, y), 1):
        caps = fit_caps(X.iloc[tr])
        Xtr, Xva = apply_caps(X.iloc[tr], caps), apply_caps(X.iloc[va], caps)
        ytr, yva = y.iloc[tr], y.iloc[va]

        inner = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
        inner_p, inner_y = [], []
        for itr, iva in inner.split(Xtr, ytr):
            _, p = fit_predict(kind, Xtr.iloc[itr], ytr.iloc[itr], Xtr.iloc[iva], params)
            inner_p.append(p); inner_y.append(ytr.iloc[iva])
        inner_p = np.concatenate(inner_p); inner_y = pd.concat(inner_y)
        th = pick_threshold(inner_p, inner_y)

        _, p_va = fit_predict(kind, Xtr, ytr, Xva, params)

        if calibrate:
            # изотоника учится на внутренних фолдах, отложенную часть она не видела
            iso = IsotonicRegression(out_of_bounds="clip").fit(inner_p, inner_y)
            p_va = iso.predict(p_va)
            th = pick_threshold(iso.predict(inner_p), inner_y)

        oof[va] = p_va
        rows.append(dict(fold=fold, threshold=round(th, 3),
                         roc_auc=roc_auc_score(yva, p_va),
                         pr_auc=average_precision_score(yva, p_va),
                         balanced_acc=balanced_accuracy_score(yva, (p_va >= th).astype(int))))

    return pd.DataFrame(rows), oof

## Метрики

In [ ]:
%%time
res_cb, oof_cb = evaluate("cb", CB_PARAMS)
res_lgb, oof_lgb = evaluate("lgb")
res_xgb, oof_xgb = evaluate("xgb")

summary = pd.DataFrame({
    name: r[["roc_auc", "pr_auc", "balanced_acc"]].agg(["mean", "std"]).round(4).stack()
    for name, r in [("CatBoost", res_cb), ("LightGBM", res_lgb), ("XGBoost", res_xgb)]
})
print("Базовый PR-AUC (доля положительного класса):", round(y.mean(), 4))
summary

In [ ]:
res_cb.round(4)

## Сравнение с первой версией проекта
Та же модель, но порог подобран на всех OOF-предсказаниях сразу, а квантили - по всему train. Разница между ними показывает стоимость утечки в данных

In [ ]:
caps_all = fit_caps(X)
X_all = apply_caps(X, caps_all)
outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_leak = np.zeros(len(X))
for tr, va in outer.split(X_all, y):
    _, oof_leak[va] = fit_predict("cb", X_all.iloc[tr], y.iloc[tr], X_all.iloc[va], CB_PARAMS)

th_leak = pick_threshold(oof_leak, y)

comparison = pd.DataFrame([
    dict(версия="v1: порог и квантили на всей выборке",
         roc_auc=roc_auc_score(y, oof_leak),
         pr_auc=average_precision_score(y, oof_leak),
         balanced_acc=balanced_accuracy_score(y, (oof_leak >= th_leak).astype(int))),
    dict(версия="v2: вложенный порог, квантили внутри фолда",
         roc_auc=res_cb.roc_auc.mean(),
         pr_auc=res_cb.pr_auc.mean(),
         balanced_acc=res_cb.balanced_acc.mean()),
]).set_index("версия").round(4)
comparison

## Ансамбль
Веса подбираются на внутренних фолдах и оцениваются на отложенных

In [ ]:
def blend_score(w, ps, t):
    p = sum(wi * pi for wi, pi in zip(w, ps))
    return balanced_accuracy_score(t, (p >= pick_threshold(p, t)).astype(int))

grid = [(a / 10, b / 10, 1 - a / 10 - b / 10)
        for a in range(11) for b in range(11 - a)]
best_w = max(grid, key=lambda w: blend_score(w, [oof_cb, oof_lgb, oof_xgb], y))
p_blend = sum(wi * pi for wi, pi in zip(best_w, [oof_cb, oof_lgb, oof_xgb]))

print("Веса CB / LGB / XGB:", best_w)
print("ROC-AUC:", round(roc_auc_score(y, p_blend), 4))
print("PR-AUC :", round(average_precision_score(y, p_blend), 4))

## ROC и PR

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5), dpi=130)
fpr, tpr, _ = roc_curve(y, oof_cb)
ax[0].plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc_score(y, oof_cb):.3f}")
ax[0].plot([0, 1], [0, 1], "--", color="gray", lw=1)
ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR"); ax[0].set_title("ROC (OOF)"); ax[0].legend()

pr, rc, _ = precision_recall_curve(y, oof_cb)
ax[1].plot(rc, pr, lw=2, label=f"PR-AUC = {average_precision_score(y, oof_cb):.3f}")
ax[1].axhline(y.mean(), ls="--", color="gray", lw=1, label=f"baseline = {y.mean():.3f}")
ax[1].set_xlabel("Recall"); ax[1].set_ylabel("Precision"); ax[1].set_title("Precision-Recall (OOF)"); ax[1].legend()
plt.tight_layout(); plt.savefig("curves.png", dpi=150); plt.show()

## SHAP

In [ ]:
import shap

caps = fit_caps(X)
X_fin = apply_caps(X, caps)
final = CatBoostClassifier(**CB_PARAMS, random_seed=SEED, verbose=False,
                           task_type="CPU", thread_count=N_JOBS)
final.fit(X_fin, y, cat_features=CAT_COLS)

expl = shap.TreeExplainer(final)
sv = expl.shap_values(X_fin)
e = shap.Explanation(values=sv, base_values=np.full(len(X_fin), expl.expected_value),
                     data=X_fin.values, feature_names=list(X_fin.columns))

shap.plots.bar(e, max_display=15, show=False)
plt.tight_layout(); plt.savefig("shap_bar.png", dpi=150); plt.show()

shap.plots.beeswarm(e, max_display=15, show=False)
plt.tight_layout(); plt.savefig("shap_beeswarm.png", dpi=150); plt.show()

## Сабмит
ID берутся из test.csv без генерации с нуля

In [ ]:
T_fin = apply_caps(T, caps)
for c in CAT_COLS:
    T_fin[c] = T_fin[c].fillna("Unknown").astype(str)

p_test = final.predict_proba(T_fin)[:, 1]
th = res_cb.threshold.median()

pd.DataFrame({"ID": test_ids, "target": (p_test >= th).astype(int)}).to_csv("submission_v2.csv", index=False)
print("порог:", th, "| доля положительных:", round((p_test >= th).mean(), 3))

# Дополнительные проверки

## 1. Нужен ли тут вообще бустинг

583 наблюдения и 18 признаков это мало.
Сравниваем при тех же условиях, но логистической регрессии дополнительно нужны импутация, масштабирование и one-hot, бустингу нет.

In [ ]:
%%time
res_lr, oof_lr = evaluate("lr")

baseline = pd.DataFrame({
    name: r[["roc_auc", "pr_auc", "balanced_acc"]].mean().round(4)
    for name, r in [("Логистическая регрессия", res_lr), ("CatBoost", res_cb)]
}).T
baseline["выигрыш CatBoost"] = (baseline.loc["CatBoost"] - baseline.loc["Логистическая регрессия"]).round(4)
baseline

In [ ]:
# разброс по фолдам для обеих моделей, если выигрыш меньше разброса, он ничего не значит
spread = pd.DataFrame({
    "CatBoost": res_cb[["roc_auc", "pr_auc", "balanced_acc"]].std().round(4),
    "Логрег": res_lr[["roc_auc", "pr_auc", "balanced_acc"]].std().round(4),
    "разница средних": (res_cb[["roc_auc", "pr_auc", "balanced_acc"]].mean()
                        - res_lr[["roc_auc", "pr_auc", "balanced_acc"]].mean()).round(4),
})
spread

## 2. Не случайность ли наши числа

Метрика получена на одном разбиении. Разброс по фолдам 0.05, значит само число зависит от того,
как легли фолды. Проверим пять разных seed и смотрим, насколько устойчив результат.

In [ ]:
%%time
def evaluate_seeds(kind, params=None, seeds=(42, 7, 13, 21, 99)):
    rows = []
    for s in seeds:
        r, _ = evaluate(kind, params, seed=s)
        rows.append(dict(seed=s,
                         roc_auc=r.roc_auc.mean(),
                         pr_auc=r.pr_auc.mean(),
                         balanced_acc=r.balanced_acc.mean()))
    return pd.DataFrame(rows)

seeds_cb = evaluate_seeds("cb", CB_PARAMS)
seeds_lr = evaluate_seeds("lr")

stability = pd.DataFrame({
    "CatBoost": seeds_cb[["roc_auc", "pr_auc", "balanced_acc"]].agg(["mean", "std"]).round(4).stack(),
    "Логрег":   seeds_lr[["roc_auc", "pr_auc", "balanced_acc"]].agg(["mean", "std"]).round(4).stack(),
})
print(seeds_cb.round(4).to_string(index=False))
stability

## 3. Можно ли читать выход модели как вероятность

Бустинг хорошо ранжирует, но его выход не обязан быть вероятностью.
Если модель говорит 0.8, доля прогрессии среди таких пациентов должна быть около 80 процентов.
Проверяем кривой калибровки и метрикой Брайера, потом чиним изотонической регрессией внутри фолда.

In [ ]:
%%time
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

res_cal, oof_cal = evaluate("cb", CB_PARAMS, calibrate=True)

calib = pd.DataFrame([
    dict(версия="без калибровки", brier=brier_score_loss(y, oof_cb),
         roc_auc=res_cb.roc_auc.mean(), pr_auc=res_cb.pr_auc.mean(),
         balanced_acc=res_cb.balanced_acc.mean()),
    dict(версия="изотоника внутри фолда", brier=brier_score_loss(y, oof_cal),
         roc_auc=res_cal.roc_auc.mean(), pr_auc=res_cal.pr_auc.mean(),
         balanced_acc=res_cal.balanced_acc.mean()),
]).set_index("версия").round(4)
calib

In [ ]:
fig, ax = plt.subplots(figsize=(5.4, 5.4), dpi=130)
ax.plot([0, 1], [0, 1], "--", color="gray", lw=1, label="идеальная калибровка")
for p, name in [(oof_cb, "без калибровки"), (oof_cal, "изотоника")]:
    frac, mean_pred = calibration_curve(y, p, n_bins=8, strategy="quantile")
    ax.plot(mean_pred, frac, "o-", lw=2, ms=6, label=f"{name}, Brier {brier_score_loss(y, p):.3f}")
ax.set_xlabel("предсказанная вероятность"); ax.set_ylabel("наблюдаемая доля")
ax.set_title("Кривая калибровки (OOF)"); ax.legend(); ax.grid(alpha=.25)
plt.tight_layout(); plt.savefig("calibration.png", dpi=150); plt.show()